对 [GPTtoLlama2](GPTtoLlama2.ipynb) 继续。考虑到运行时的便利，以及上个笔记本已经很长，重新分出来一个笔记本。

## Llama2toLlama3

在 OpenAI 发布 GPT-4 不久后，Meta AI 发布了 Llama3，一度以开源模型追上了闭源模型的性能，在 2024 年成为开源模型的首选。针对于 Llama2 到 Llama3 的升级，最大的改动是将 MHA（传统多头注意力） 更改成了 GQA（组查询注意力），以及在训练数据上的处理和模型规模上的扩大，最大的模型达到 405B。

Meta AI 开源提供了 8B 70B 405B 三个规模的模型，大多数情况下 70B 是性能和花费均衡的选择。notebook 最后我们会加载 8B 模型进行演示。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

### 复用 Llama2 中实现的组件

正如上图，Llama3 的架构与 Llama2 差异很小。我们可以复用 [GPTtoLlama2](GPTtoLlama2.ipynb) 中实现的模块。

In [ ]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    
    def import_definitions_from_notebook(fullname, names):
        """
        从当前目录下的 fullname.ipynb 中加载指定名字的定义。

        fullname: 不带 .ipynb 后缀的 notebook 文件名。
        names: 需要加载的函数名或类名字符串列表。
        """
        current_dir = os.getcwd()
        path = os.path.join(current_dir, fullname + ".ipynb")
        path = os.path.normpath(path)

        # 加载笔记本文件
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")
        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # 动态创建一个模块对象，并把它注册到 Python 的模块缓存 sys.modules 中
        # 等价于创建了一个空的命名空间，名字是 fullname，作为执行 notebook 代码的命名空间
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # 遍历所有 notebook 块
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # 检查 code 块中是否包含定义函数或类的代码
                    # 如果出现，就执行整个代码单元
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "GPTtoLlama2"
    names = ["precompute_rope_params", "compute_rope", "SiLU", "FeedForward", "RMSNorm", "MultiHeadAttention"]

    return import_definitions_from_notebook(fullname, names)